In [1]:
#!/usr/bin/env python3
import sys 
sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/waveformArchive/gcc_build')
#sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group1/bbaker/templateMatchingSource/rtseis/notchpeak4_gcc83_build/')
sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/mlmodels/intel_cpu_build')
sys.path.insert(0, '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/mlmodels/features/np4_build')
import h5py
import pyWaveformArchive as pwa 
import pyuussmlmodels as uuss
import pyuussFeatures as pf
# import libpyrtseis as rtseis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

import glob

# P-arrivals

In [2]:
archive_dir = '/uufs/chpc.utah.edu/common/home/koper-group4/bbaker/waveformArchive/archives/'
h5_archive_files = glob.glob(archive_dir + '/archive_????.h5')
catalog_dir = '/uufs/chpc.utah.edu/common/home/koper-group3/alysha/ben_catalogs/20240220'
arrival_catalog_1c = f'{catalog_dir}/currentEarthquakeArrivalInformation1CWithGains.csv'
arrival_catalog_3c = f'{catalog_dir}/currentEarthquakeArrivalInformation3CWithGains.csv'
startdate = datetime(2002, 1, 1, tzinfo=timezone.utc).timestamp()
print(f'Using events occuring on or after {startdate}')

print("Loading arrival catalog...")
arrival_catalog_1c_df = pd.read_csv(arrival_catalog_1c, dtype = {'location' : object})
arrival_catalog_3c_df = pd.read_csv(arrival_catalog_3c, dtype = {'location' : object})
print(arrival_catalog_1c_df.dtypes)
arrival_catalog_3c_df = arrival_catalog_3c_df[arrival_catalog_1c_df.columns] # Only keep relevant columns
arrival_catalog_df = pd.concat([arrival_catalog_1c_df, arrival_catalog_3c_df], ignore_index = True)
# Regress on Ml
arrival_catalog_df = arrival_catalog_df[ (arrival_catalog_df.phase == 'P') &
                                            (arrival_catalog_df.magnitude_type == 'l') &
                                            (arrival_catalog_df.origin_time >= startdate)]
# # Focus on Yellowstone
arrival_catalog_df = arrival_catalog_df[ (arrival_catalog_df.event_lat > 44) &
                                            (arrival_catalog_df.event_lat < 45.167) &
                                            (arrival_catalog_df.event_lon > -111.333) &
                                            (arrival_catalog_df.event_lon < -109.75) ]

Using events occuring on or after 1009843200.0
Loading arrival catalog...
evid                          int64
network                      object
station                      object
location                     object
channelz                     object
phase                        object
arrival_id                    int64
arrival_time                float64
pick_quality                float64
first_motion                  int64
take_off_angle                int64
source_receiver_distance    float64
source_receiver_azimuth     float64
travel_time_residual        float64
receiver_lat                float64
receiver_lon                float64
receiver_elev               float64
event_lat                   float64
event_lon                   float64
event_depth                 float64
origin_time                 float64
magnitude                   float64
magnitude_type               object
rflag                        object
gain_z                      float64
gain_units                

In [25]:
arrival_catalog_df.event_depth

198       4.43
199       4.43
200       4.43
201       7.69
202       7.69
          ... 
556700    9.12
556702    9.12
556703    9.12
556705    9.12
556707    9.12
Name: event_depth, Length: 108566, dtype: float64

In [3]:
arrival_catalog_df["location"].value_counts()

location
01    94223
      11946
00     2397
Name: count, dtype: int64

In [4]:
np.where((arrival_catalog_df["location"] != "01") & (arrival_catalog_df["location"] != "00"))

(array([ 15678,  15679,  15680, ..., 107669, 107686, 107696]),)

In [5]:
arrival_catalog_df.iloc[15678]["location"] == "  "

True

In [6]:
arrival_catalog_df["gain_units"].value_counts()

gain_units
DU/M/S       108090
DU/M/S**2       475
DU/V              1
Name: count, dtype: int64

# Make df of unique channel gains

In [7]:
gains_df = arrival_catalog_df[["network", "station", "channelz", "location", "gain_z", "gain_units"]].drop_duplicates().sort_values(["network", "station", "channelz", "location"])
gains_df.head()

,network,station,channelz,location,gain_z,gain_units
316621,GS,ID05,HHZ,00,468849000.0,DU/V
316620,GS,ID08,HHZ,00,468849000.0,DU/M/S
307896,IE,DVCI,HHZ,,503831000.0,DU/M/S
63060,IE,ECRI,EHZ,,86004000.0,DU/M/S
581,IE,ECRI,EHZ,01,86004000.0,DU/M/S


In [8]:
row = gains_df.iloc[0]
print(row)
arrival_catalog_df[np.all(arrival_catalog_df[["network", "station", "channelz", "location", "gain_z", "gain_units"]] == row, axis=1)].drop_duplicates(["network", "station", "channelz", "location", "gain_z", "gain_units"])

network                GS
station              ID05
channelz              HHZ
location               00
gain_z        468849000.0
gain_units           DU/V
Name: 316621, dtype: object


,evid,network,station,location,channelz,phase,arrival_id,arrival_time,pick_quality,first_motion,...,origin_time,magnitude,magnitude_type,rflag,gain_z,gain_units,low_freq_corners_z,high_freq_corners_z,channel_dip_z,channel_azimuth_z
316621,60246847,GS,ID05,00,HHZ,P,10325288,1.505627e+09,0.75,0,...,1.505627e+09,3.15,l,F,468849000.0,DU/V,40.0,0.008305,-90.0,0.0


In [9]:
gains_mindates = []
gains_maxdates = []
for _, row in gains_df.iterrows():
    t_df = arrival_catalog_df[np.all(arrival_catalog_df[["network", "station", "channelz", "location", "gain_z", "gain_units"]] == row, axis=1)]
    mindate = datetime.fromtimestamp(t_df["arrival_time"].min(), tz=timezone.utc)
    maxdate = datetime.fromtimestamp(t_df["arrival_time"].max(), tz=timezone.utc)
    gains_mindates.append(mindate)
    gains_maxdates.append(maxdate)

gains_df["mindate"] = gains_mindates
gains_df["maxdate"] = gains_maxdates
gains_df.head()

,network,station,channelz,location,gain_z,gain_units,mindate,maxdate
316621,GS,ID05,HHZ,00,468849000.0,DU/V,2017-09-17 05:39:16.766003+00:00,2017-09-17 05:39:16.766003+00:00
316620,GS,ID08,HHZ,00,468849000.0,DU/M/S,2017-09-17 05:39:14.359248+00:00,2017-09-17 05:39:14.359248+00:00
307896,IE,DVCI,HHZ,,503831000.0,DU/M/S,2017-08-01 15:16:23.227715+00:00,2017-08-01 15:16:23.227715+00:00
63060,IE,ECRI,EHZ,,86004000.0,DU/M/S,2017-06-12 22:55:28.388086+00:00,2018-04-05 13:23:03.382463+00:00
581,IE,ECRI,EHZ,01,86004000.0,DU/M/S,2012-10-15 02:55:18.977618+00:00,2014-04-03 11:09:54.386802+00:00


# Load in the database channel info

In [10]:
db_gains_df = pd.read_csv("/uufs/chpc.utah.edu/common/home/u1072028/PycharmProjects/seis-proc-db/data_files/db_channel_info.csv",  dtype = {'location' : object}).sort_values(["network", "station", "seed_code", "location"])
db_gains_df["location"] = db_gains_df["location"].fillna("  ")
db_gains_df.head()

,channel_id,network,station,location,seed_code,channel_samp_rate,sensit_units,sensit_freq,sensit_val,gain_vel,receiver_lat,receiver_lon,channel_azimuth,channel_ondate,channel_offdate
3,958,IW,FLWY,00,BH1,40.0,m/s,0.05,1.145990e+09,1.145993e+09,44.083002,-110.699888,0.0,2013-07-25 14:46:00,2016-10-22 04:00:00
4,568,IW,FLWY,00,BH1,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,0.0,2016-10-22 04:00:00,2020-01-30 20:00:00
5,377,IW,FLWY,00,BH1,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,0.0,2020-01-30 20:00:00,2021-08-29 18:00:00
6,4,IW,FLWY,00,BH1,40.0,m/s,0.02,2.939200e+08,2.939199e+08,44.083002,-110.699888,2.8,2021-08-29 18:00:00,NaN
7,959,IW,FLWY,00,BH2,40.0,m/s,0.05,1.145990e+09,1.145993e+09,44.083002,-110.699888,90.0,2013-07-25 14:46:00,2016-10-22 04:00:00


In [11]:
#db_gains_df["simple_gain_vel"] = db_gains_df.apply(lambda x: x["sensit_val"]*(2*np.pi*x["sensit_freq"])**-1 if x["sensit_units"] in ["m", "M"] else x["sensit_val"], axis=1)
simple_gain_vels = []
for _, row in db_gains_df.iterrows():
    sensit_val = row["sensit_val"]
    if row["sensit_units"] in ["m", "M"]:
        sensit_val = row["sensit_val"]*(2*np.pi*row["sensit_freq"])**-1
    elif row["sensit_units"] in ["m/s**2", "M/S**2"]:
        sensit_val = row["sensit_val"]*(2*np.pi*row["sensit_freq"])

    simple_gain_vels.append(sensit_val)
db_gains_df["simple_gain_vel"] = simple_gain_vels

# Compare gainz

In [12]:
for i, row in gains_df.iterrows():
    matching_db_df = db_gains_df[(db_gains_df["network"] == row["network"]) & 
                                 (db_gains_df["station"] == row["station"]) & 
                                 (db_gains_df["seed_code"] == row["channelz"])]
                                 # & (db_gains_df["sensit_units"].isin(["m", "M"]))]
                                 # & (db_gains_df["location"] == row["location"])]
    
    if len(matching_db_df) > 0:
        diffs = abs(matching_db_df["simple_gain_vel"] - row["gain_z"])
        min_diff_i = np.argmin(diffs)
        min_diff = diffs.iloc[min_diff_i]
        min_db_row = matching_db_df.iloc[min_diff_i]
        diff1 = abs(min_db_row["gain_vel"] - row["gain_z"])

        print(f'{row["network"]}.{row["station"]}.{row["channelz"]}.{row["location"]:5s} {row["gain_units"]} {min_db_row["sensit_units"]:3s} {min_diff:0.2f} {diff1:.2f} {min_diff-diff1:.2f} {(min_diff/row["gain_z"])*100:.2f}%  {i}')

IW.FLWY.BHZ.00    DU/M/S m/s 0.00 675.19 -675.19 0.00%  186336
IW.FLWY.BHZ.00    DU/M/S m/s 7800000.00 7800148.53 -148.53 2.59%  218130
IW.FLWY.BHZ.01    DU/M/S m/s 112660000.00 112656841.53 3158.47 8.95%  177740
IW.IMW.BHZ.00    DU/M/S m/s 112650000.00 112657709.45 -7709.45 8.95%  186450
IW.IMW.BHZ.01    DU/M/S m/s 112650000.00 112657709.45 -7709.45 8.95%  177857
MB.QLMT.EHZ.01    DU/M/S m/s 0.00 1012442166928.15 -1012442166928.15 0.00%  514
MB.QLMT.EHZ.01    DU/M/S m/s 0.00 8098022779022.01 -8098022779022.01 0.00%  892
MB.QLMT.EHZ.01    DU/M/S m/s 0.00 16196045558044.02 -16196045558044.02 0.00%  3495
MB.TPMT.EHZ.01    DU/M/S m/s 0.00 16865852934.60 -16865852934.60 0.00%  559
MB.TPMT.EHZ.01    DU/M/S m/s 0.00 269803096157.33 -269803096157.33 0.00%  15547
MB.TPMT.EHZ.01    DU/M/S m/s 0.00 134901547578.67 -134901547578.67 0.00%  21251
PB.B206.EHZ.      DU/M/S M/S 0.00 10219774.27 -10219774.27 0.00%  185994
PB.B206.EHZ.01    DU/M/S M/S 0.00 10219774.27 -10219774.27 0.00%  177537
PB.B207.

In [13]:
net = "WY"
stat = "YTP"
loc = "00"
chan = "EHZ"
gains_df[(gains_df["network"] == net) & 
            (gains_df["station"] == stat) & 
            #(gains_df["location"] == loc) & 
            (gains_df["channelz"] == chan)].sort_values("mindate")

,network,station,channelz,location,gain_z,gain_units,mindate,maxdate
200,WY,YTP,EHZ,01,6.638468e+09,DU/M/S,2012-10-15 04:02:28.157829+00:00,2013-08-21 09:23:14.190322+00:00
2838,WY,YTP,EHZ,01,8.298078e+08,DU/M/S,2013-09-01 09:45:42.177610+00:00,2023-12-17 04:31:43.530981+00:00


In [14]:
db_gains_df[(db_gains_df["network"] == net) & 
            (db_gains_df["station"] == stat) & 
            # (db_gains_df["location"] == loc) & 
            (db_gains_df["seed_code"] == chan)].sort_values("channel_ondate")

,channel_id,network,station,location,seed_code,channel_samp_rate,sensit_units,sensit_freq,sensit_val,gain_vel,receiver_lat,receiver_lon,channel_azimuth,channel_ondate,channel_offdate,simple_gain_vel
306,1507,WY,YTP,,EHZ,100.0,m,5.0,1.302940e+10,4.147370e+08,44.39183,-110.285,0.0,1994-12-23 00:00:00,2010-04-30 23:59:59,4.147387e+08
307,1211,WY,YTP,,EHZ,100.0,m,5.0,2.085490e+11,6.638468e+09,44.39183,-110.285,0.0,2010-05-01 00:00:00,2013-03-31 23:59:59,6.638321e+09
308,105,WY,YTP,01,EHZ,100.0,m/s,5.0,8.301530e+08,8.301511e+08,44.39183,-110.285,0.0,2013-04-01 00:00:00,NaN,8.301530e+08


In [15]:
# for i, row in db_gains_df.iterrows():
#     matching_bb_df = gains_df[(gains_df["network"] == row["network"]) & 
#                               (gains_df["station"] == row["station"]) & 
#                               (gains_df["channelz"] == row["seed_code"])]
#                                  # & (db_gains_df["sensit_units"].isin(["m", "M"]))]
#                                  # & (db_gains_df["location"] == row["location"])]
    
#     if len(matching_bb_df) > 0:
#         diffs = abs(matching_bb_df["gain_z"] - row["gain_vel"])
#         # if row["sensit_units"] in ["m", "M"]:
#         #     diffs = abs(matching_bb_df["gain_z"] - row["gain_vel"])
#         # else:
#         #     diffs = abs(matching_bb_df["gain_z"] - row["sensit_val"])
#         min_diff_i = np.argmin(diffs)
#         min_diff = diffs.iloc[min_diff_i]
#         min_db_row = matching_bb_df.iloc[min_diff_i]

#         print(f'{row["network"]}.{row["station"]}.{row["seed_code"]}.{row["location"]:5s} {row["sensit_units"]} {min_db_row["gain_units"]:3s} {min_diff:0.2f} {(min_diff/min_db_row["gain_z"])*100:.2f}% {i}')

# Save matching gain info

In [16]:
matching_gains = []
matching_gains_units = []
for i, row in db_gains_df.iterrows():
    chan_ondate = datetime.strptime(row["channel_ondate"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    chan_offdate = row["channel_offdate"]
    if type(chan_offdate) == str:    
        chan_offdate = datetime.strptime(chan_offdate, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    else:
        chan_offdate = None
    #print(chan_ondate, chan_offdate)
    matching_bb_df = gains_df[(gains_df["network"] == row["network"]) & 
                              (gains_df["station"] == row["station"]) & 
                              (gains_df["channelz"] == row["seed_code"]) &
                              (chan_offdate is None or gains_df["mindate"] <= chan_offdate) &
                              (gains_df["maxdate"] >= chan_ondate)]
                                 # & (db_gains_df["sensit_units"].isin(["m", "M"]))]
                                 # & (db_gains_df["location"] == row["location"])]
    matching_gain = None
    matching_gain_units = None
    if len(matching_bb_df) > 0:
        uniq_vals = matching_bb_df["gain_z"].drop_duplicates().values
        print(row["network"], row["station"], row["seed_code"], row["location"], row["gain_vel"], chan_ondate, chan_offdate, len(matching_bb_df), uniq_vals)
        if len(uniq_vals) > 1:
            date_diffs = matching_bb_df["maxdate"] - chan_ondate
            matching_bb_df = matching_bb_df.iloc[np.argmax(date_diffs):np.argmax(date_diffs)+1]
            print("*", row["network"], row["station"], row["seed_code"], row["location"], row["gain_vel"], chan_ondate, chan_offdate, matching_bb_df["gain_z"].values)
        
        matching_gain = matching_bb_df["gain_z"].values[0]
        matching_gain_units = matching_bb_df["gain_units"].values[0]
        #print(matching_bb_df)

    matching_gains.append(matching_gain)
    matching_gains_units.append(matching_gain_units)

IW FLWY BHZ 00 1145993158.470899 2013-07-25 14:46:00+00:00 2016-10-22 04:00:00+00:00 1 [1.25865e+09]
IW FLWY BHZ 00 462428324.8058124 2016-10-22 04:00:00+00:00 2020-01-30 20:00:00+00:00 1 [4.62429e+08]
IW FLWY BHZ 00 462428324.8058124 2020-01-30 20:00:00+00:00 2021-08-29 18:00:00+00:00 1 [4.62429e+08]
IW FLWY BHZ 00 293919851.46844804 2021-08-29 18:00:00+00:00 None 1 [3.0172e+08]
IW FLWY BHZ    1145993158.470899 2007-03-16 16:46:00+00:00 2013-07-25 14:46:00+00:00 1 [1.25865e+09]
IW IMW BHZ 00 1145993158.470899 2013-07-25 15:11:00+00:00 2023-09-15 17:00:00+00:00 2 [1.25865e+09]
IW IMW BHZ    1145993158.470899 2006-08-03 20:00:00+00:00 2013-07-25 15:11:00+00:00 1 [1.25865e+09]
MB QLMT EHZ 01 16200374598044.018 2019-03-21 16:40:00+00:00 None 1 [4.32904e+09]
MB QLMT EHZ    1012712731928.1469 2003-06-10 18:00:00+00:00 2013-09-06 18:00:00+00:00 1 [2.70565e+08]
MB QLMT EHZ    8100187299022.009 2013-09-06 18:00:00+00:00 2014-10-23 16:55:00+00:00 1 [2.16452e+09]
MB QLMT EHZ    16200374598044.01

In [17]:
db_gains_df["featmag_gain"] = matching_gains
db_gains_df["featmag_gain_units"] = matching_gains_units

In [18]:
for i, row in db_gains_df[~np.isnan(db_gains_df["featmag_gain"])].iterrows():
    gain_perc_err = (abs(row["gain_vel"] - row["featmag_gain"])/row["featmag_gain"])*100
    simple_gain_perc_err = (abs(row["simple_gain_vel"] - row["featmag_gain"])/row["featmag_gain"])*100
    print(f'{row["network"]}.{row["station"]}.{row["seed_code"]}.{row["location"]:5s}{ row["channel_ondate"]}-{row["channel_offdate"]} {gain_perc_err:.4f}% {simple_gain_perc_err:.4f}%')

IW.FLWY.BHZ.00   2013-07-25 14:46:00-2016-10-22 04:00:00 8.9506% 8.9509%
IW.FLWY.BHZ.00   2016-10-22 04:00:00-2020-01-30 20:00:00 0.0001% 0.0000%
IW.FLWY.BHZ.00   2020-01-30 20:00:00-2021-08-29 18:00:00 0.0001% 0.0000%
IW.FLWY.BHZ.00   2021-08-29 18:00:00-nan 2.5852% 2.5852%
IW.FLWY.BHZ.     2007-03-16 16:46:00-2013-07-25 14:46:00 8.9506% 8.9509%
IW.IMW.BHZ.00   2013-07-25 15:11:00-2023-09-15 17:00:00 8.9506% 8.9509%
IW.IMW.BHZ.     2006-08-03 20:00:00-2013-07-25 15:11:00 8.9506% 8.9509%
MB.QLMT.EHZ.01   2019-03-21 16:40:00-nan 374125.5696% 0.0000%
MB.QLMT.EHZ.     2003-06-10 18:00:00-2013-09-06 18:00:00 374195.5415% 0.0000%
MB.QLMT.EHZ.     2013-09-06 18:00:00-2014-10-23 16:55:00 374125.5696% 0.0000%
MB.QLMT.EHZ.     2014-10-23 16:55:00-2019-03-21 16:40:00 374125.5696% 0.0000%
MB.TPMT.EHZ.01   2019-03-21 16:40:00-nan 48243.9953% 0.0000%
MB.TPMT.EHZ.     2008-07-15 18:10:00-2013-09-06 18:00:00 48252.9481% 0.0000%
MB.TPMT.EHZ.     2013-09-06 18:00:00-2014-10-23 16:55:00 48243.9088% 0.00

In [19]:
net = "IW"
stat = "FLWY"
loc = "00"
chan = "BHZ"
gains_df[(gains_df["network"] == net) & 
            (gains_df["station"] == stat) & 
            #(gains_df["location"] == loc) & 
            (gains_df["channelz"] == chan)].sort_values("mindate")

,network,station,channelz,location,gain_z,gain_units,mindate,maxdate
177740,IW,FLWY,BHZ,01,1.258650e+09,DU/M/S,2012-10-15 02:47:29.962205+00:00,2016-08-14 01:24:05.391083+00:00
186336,IW,FLWY,BHZ,00,4.624290e+08,DU/M/S,2016-10-23 05:32:49.845000+00:00,2021-08-20 17:00:43.626704+00:00
218130,IW,FLWY,BHZ,00,3.017200e+08,DU/M/S,2021-09-08 17:01:36.960548+00:00,2023-12-17 04:29:30.569849+00:00


In [20]:
db_gains_df[(db_gains_df["network"] == net) & 
            (db_gains_df["station"] == stat) & 
            # (db_gains_df["location"] == loc) & 
            (db_gains_df["seed_code"] == chan)].sort_values("channel_ondate")

,channel_id,network,station,location,seed_code,channel_samp_rate,sensit_units,sensit_freq,sensit_val,gain_vel,receiver_lat,receiver_lon,channel_azimuth,channel_ondate,channel_offdate,simple_gain_vel,featmag_gain,featmag_gain_units
13,1170,IW,FLWY,,BHZ,40.0,m/s,0.05,1.145990e+09,1.145993e+09,44.083002,-110.699888,0.0,2007-03-16 16:46:00,2013-07-25 14:46:00,1.145990e+09,1.258650e+09,DU/M/S
14,960,IW,FLWY,00,BHZ,40.0,m/s,0.05,1.145990e+09,1.145993e+09,44.083002,-110.699888,0.0,2013-07-25 14:46:00,2016-10-22 04:00:00,1.145990e+09,1.258650e+09,DU/M/S
15,570,IW,FLWY,00,BHZ,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,0.0,2016-10-22 04:00:00,2020-01-30 20:00:00,4.624290e+08,4.624290e+08,DU/M/S
16,379,IW,FLWY,00,BHZ,40.0,m/s,0.02,4.624290e+08,4.624283e+08,44.083002,-110.699888,0.0,2020-01-30 20:00:00,2021-08-29 18:00:00,4.624290e+08,4.624290e+08,DU/M/S
17,6,IW,FLWY,00,BHZ,40.0,m/s,0.02,2.939200e+08,2.939199e+08,44.083002,-110.699888,0.0,2021-08-29 18:00:00,NaN,2.939200e+08,3.017200e+08,DU/M/S


In [21]:
db_gains_df["sensit_units"].value_counts()

sensit_units
m         227
m/s       164
M/S        48
V          30
m/s**2      3
Name: count, dtype: int64

In [22]:
db_gains_df[db_gains_df["sensit_units"] == "m/s**2"] #.iloc[0]

,channel_id,network,station,location,seed_code,channel_samp_rate,sensit_units,sensit_freq,sensit_val,gain_vel,receiver_lat,receiver_lon,channel_azimuth,channel_ondate,channel_offdate,simple_gain_vel,featmag_gain,featmag_gain_units
47,17,RE,JKLK1,20,HH1,100.0,m/s**2,1.0,1.796582e+07,1.128723e+08,43.86147,-110.59174,147.6,2019-08-28 00:00:00,NaN,1.128826e+08,NaN,None
48,18,RE,JKLK1,20,HH2,100.0,m/s**2,1.0,1.796582e+07,1.128723e+08,43.86147,-110.59174,237.6,2019-08-28 00:00:00,NaN,1.128826e+08,NaN,None
49,19,RE,JKLK1,20,HHZ,100.0,m/s**2,1.0,1.796582e+07,1.128723e+08,43.86147,-110.59174,0.0,2019-08-28 00:00:00,NaN,1.128826e+08,NaN,None


In [23]:
db_gains_df.to_csv("../files/db_featmag_gains_P.csv", index=False)